# Nave Aurora Siger - Simulação Temporal de Pré-Decolagem

Este notebook acompanha a versão atual do projeto. Ele usa a mesma lógica temporal dos scripts Python, com até 61 leituras de telemetria, limites de segurança em unidades consistentes, correções automáticas e interrupção por falha persistente.


## 1. Configuração da simulação

O cenário principal usado para demonstrar o resultado esperado é `SUCESSO_DIRETO`. Os demais cenários são comparados ao final.


In [1]:
import random

import pandas as pd

from main_interativo import calcular_energia, decidir_status
from simulacao_decolagem import (
    PARAMETROS_SEGURANCA,
    analisar_cenario,
    converter_para_telemetria_resumida,
    gerar_telemetria_cenario,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


In [2]:
cenario = "SUCESSO_DIRETO"
leituras = gerar_telemetria_cenario(cenario)
analise = analisar_cenario(cenario, leituras)
telemetria_final = converter_para_telemetria_resumida(analise["telemetria_final"])
resultado_energia = calcular_energia(telemetria_final)
status, motivo_aborto, verificacoes = decidir_status(analise, resultado_energia)

primeira = leituras[0]
ultima = leituras[-1]
print(f"Cenário: {cenario}")
print(f"Leituras geradas: {len(leituras)}")
print(f"Tempo inicial: T+{primeira['tempo_decolagem']}s")
print(f"Tempo final: T+{ultima['tempo_decolagem']}s")
print(f"Energia inicial: {primeira['nivel_energia']:.2f}%")
print(f"Energia final: {ultima['nivel_energia']:.2f}%")
print(f"Status final: {status}")


Cenário: SUCESSO_DIRETO
Leituras geradas: 61
Tempo inicial: T+0s
Tempo final: T+60s
Energia inicial: 100.00%
Energia final: 94.00%
Status final: PRONTO PARA DECOLAR


## 2. Telemetria temporal

Cada linha representa uma leitura da contagem de pré-decolagem. Abaixo aparecem as últimas leituras do cenário seguro.


In [3]:
telemetria_df = pd.DataFrame(leituras)
colunas = [
    "tempo_decolagem",
    "temperatura_interna",
    "temperatura_externa",
    "vibracao_estrutural_g",
    "nivel_energia",
    "pressao_lh2_psi",
    "pressao_lox_psi",
    "motor_ok",
    "navegacao_ok",
    "comunicacao_ok",
    "sistema_eletrico_ok",
    "resfriamento_ativo",
    "pressurizacao_ativa",
]
print(telemetria_df[colunas].tail(8).to_string(index=False))


 tempo_decolagem  temperatura_interna  temperatura_externa  vibracao_estrutural_g  nivel_energia  pressao_lh2_psi  pressao_lox_psi  motor_ok  navegacao_ok  comunicacao_ok  sistema_eletrico_ok  resfriamento_ativo  pressurizacao_ativa
              53                23.53                17.73                  1.395           94.7           125.65           127.59      True          True            True                 True               False                False
              54                23.54                17.73                  1.410           94.6           125.70           127.62      True          True            True                 True               False                False
              55                23.55                17.73                  1.425           94.5           125.75           127.65      True          True            True                 True               False                False
              56                23.56                17.72          

## 3. Parâmetros de segurança

Os limites abaixo são os mesmos utilizados pelo programa para aprovar ou reprovar cada leitura.


In [4]:
limites_df = pd.DataFrame(
    [
        {"parametro": parametro, "valor_minimo": limites[0], "valor_maximo": limites[1]}
        for parametro, limites in PARAMETROS_SEGURANCA.items()
    ]
)
print(limites_df.to_string(index=False))


            parametro  valor_minimo  valor_maximo
  temperatura_interna            18          27.0
  temperatura_externa            10          34.0
vibracao_estrutural_g             0           1.5
        nivel_energia            80         100.0
      pressao_lh2_psi           120         130.0
      pressao_lox_psi           120         130.0


## 4. Verificações finais e decisão

Além da leitura final, o programa interrompe a contagem se uma falha permanecer por seis leituras consecutivas.


In [5]:
print("Verificações finais")
print("-" * 45)
for item, aprovado in verificacoes.items():
    print(f"{item}: {'OK' if aprovado else 'FALHA'}")

print("\nDecisão final")
print("-" * 45)
print(status)
if motivo_aborto:
    print(f"Motivo: {motivo_aborto}")
else:
    print("Nenhum motivo de aborto identificado.")


Verificações finais
---------------------------------------------
Temperatura interna: OK
Temperatura externa: OK
Vibração estrutural: OK
Nível de energia: OK
Pressão LH2: OK
Pressão LOX: OK
Integridade estrutural: OK
Motor: OK
Navegação: OK
Comunicação: OK
Sistema elétrico: OK
Energia após decolagem: OK

Decisão final
---------------------------------------------
PRONTO PARA DECOLAR
Nenhum motivo de aborto identificado.


## 5. Análise energética

A energia é calculada com a carga final da telemetria do cenário `SUCESSO_DIRETO`, que chega a 94% em `T+60s`.


In [6]:
print(f"Energia disponível: {resultado_energia['energia_disponivel_kwh']:.2f} kWh")
print(f"Perdas energéticas: {resultado_energia['perdas_kwh']:.2f} kWh")
print(f"Energia restante após decolagem: {resultado_energia['energia_restante_kwh']:.2f} kWh")
print(f"Autonomia inicial estimada: {resultado_energia['autonomia_horas']:.2f} horas")


Energia disponível: 1128.00 kWh
Perdas energéticas: 67.68 kWh
Energia restante após decolagem: 800.32 kWh
Autonomia inicial estimada: 9.42 horas


## 6. Análise assistida por IA

A decisão automática do programa utiliza regras e faixas de segurança predefinidas. A inteligência artificial foi utilizada como apoio para classificar os dados, interpretar possíveis anomalias e elaborar sugestões de risco. A decisão final continua sujeita às regras de segurança e à supervisão humana. Não se afirma que o programa possui uma IA própria ou um modelo treinado com dados de missões reais.


In [7]:
classificacao_dados = {
    "temperatura_interna": "dado numérico real",
    "temperatura_externa": "dado numérico real",
    "vibracao_estrutural_g": "dado numérico real",
    "nivel_energia": "dado numérico real",
    "pressao_lh2_psi": "dado numérico real",
    "pressao_lox_psi": "dado numérico real",
    "integridade_estrutural_ok": "dado lógico/binário",
    "motor_ok": "dado lógico/binário",
    "navegacao_ok": "dado lógico/binário",
    "comunicacao_ok": "dado lógico/binário",
    "sistema_eletrico_ok": "dado lógico/binário",
}

anomalias = [item for item, aprovado in verificacoes.items() if not aprovado]

print("Classificação dos dados")
print("-" * 45)
for dado, classificacao in classificacao_dados.items():
    print(f"{dado}: {classificacao}")

print("\nPossíveis anomalias")
print("-" * 45)
if anomalias:
    for anomalia in anomalias:
        print(f"Atenção: {anomalia} fora do padrão seguro")
else:
    print("Nenhuma anomalia crítica encontrada no cenário SUCESSO_DIRETO.")


Classificação dos dados
---------------------------------------------
temperatura_interna: dado numérico real
temperatura_externa: dado numérico real
vibracao_estrutural_g: dado numérico real
nivel_energia: dado numérico real
pressao_lh2_psi: dado numérico real
pressao_lox_psi: dado numérico real
integridade_estrutural_ok: dado lógico/binário
motor_ok: dado lógico/binário
navegacao_ok: dado lógico/binário
comunicacao_ok: dado lógico/binário
sistema_eletrico_ok: dado lógico/binário

Possíveis anomalias
---------------------------------------------
Nenhuma anomalia crítica encontrada no cenário SUCESSO_DIRETO.


## 7. Comparação entre cenários

A tabela resume os cenários disponíveis no programa.


In [8]:
resumos = []
for nome in ["SUCESSO_DIRETO", "ERRO_CORRIGIDO", "FALHA_CRITICA"]:
    leituras_cenario = gerar_telemetria_cenario(nome)
    analise_cenario = analisar_cenario(nome, leituras_cenario)
    telemetria_cenario = converter_para_telemetria_resumida(analise_cenario["telemetria_final"])
    energia_cenario = calcular_energia(telemetria_cenario)
    status_cenario, motivo_cenario, _ = decidir_status(analise_cenario, energia_cenario)
    resumos.append(
        {
            "cenario": nome,
            "leituras": len(leituras_cenario),
            "tempo_final": f"T+{leituras_cenario[-1]['tempo_decolagem']}s",
            "energia_final_%": telemetria_cenario["nivel_energia_percentual"],
            "pressao_media_psi": telemetria_cenario["pressao_media_tanques_psi"],
            "status": status_cenario,
            "motivo_aborto": motivo_cenario or "-",
        }
    )

random.seed(7)
leituras_customizadas = gerar_telemetria_cenario("CUSTOMIZADO")
analise_customizada = analisar_cenario("CUSTOMIZADO", leituras_customizadas)
telemetria_customizada = converter_para_telemetria_resumida(analise_customizada["telemetria_final"])
energia_customizada = calcular_energia(telemetria_customizada)
status_customizado, motivo_customizado, _ = decidir_status(analise_customizada, energia_customizada)
resumos.append(
    {
        "cenario": "CUSTOMIZADO (semente 7)",
        "leituras": len(leituras_customizadas),
        "tempo_final": f"T+{leituras_customizadas[-1]['tempo_decolagem']}s",
        "energia_final_%": telemetria_customizada["nivel_energia_percentual"],
        "pressao_media_psi": telemetria_customizada["pressao_media_tanques_psi"],
        "status": status_customizado,
        "motivo_aborto": motivo_customizado or "-",
    }
)

print(pd.DataFrame(resumos).to_string(index=False))


                cenario  leituras tempo_final  energia_final_%  pressao_media_psi              status    motivo_aborto
         SUCESSO_DIRETO        61       T+60s            94.00             126.90 PRONTO PARA DECOLAR                -
         ERRO_CORRIGIDO        61       T+60s            93.40             125.47 PRONTO PARA DECOLAR                -
          FALHA_CRITICA        41       T+40s            95.50             126.15  DECOLAGEM ABORTADA Sistema elétrico
CUSTOMIZADO (semente 7)        22       T+21s            95.43             127.34  DECOLAGEM ABORTADA      Comunicação


## 7. Reflexão crítica

### Introdução

O projeto de verificação de condições para uma decolagem permite refletir sobre a responsabilidade no uso da tecnologia. A análise da telemetria, a avaliação energética e o apoio da inteligência artificial devem estar associados à segurança, à transparência e ao uso consciente dos recursos. Essa perspectiva se relaciona ao tripé da sustentabilidade, que considera as dimensões econômica, social e ambiental, e aos princípios de governança presentes no ESG.

### Ética e responsabilidade

No aspecto ético, a segurança deve orientar os critérios de autorização ou cancelamento da decolagem. Informações sobre temperatura, integridade estrutural, energia e pressão precisam ser verificadas, pois dados incorretos ou incompletos podem comprometer a decisão. Também é necessário apresentar os motivos do resultado, permitindo compreender quais condições foram consideradas inadequadas. A inteligência artificial pode auxiliar na identificação de anomalias, mas suas sugestões devem ser avaliadas, mantendo a responsabilidade humana. Em uma aplicação real, a simulação precisaria passar por validações técnicas rigorosas antes de integrar uma operação.

### Impacto social da exploração espacial

Quanto ao impacto social, a exploração espacial pode contribuir para pesquisas científicas, comunicação e monitoramento ambiental. Entretanto, esses benefícios precisam ser avaliados junto aos custos, aos riscos e à sua distribuição na sociedade. Uma atuação socialmente responsável deve considerar tanto as pessoas envolvidas nas operações quanto as comunidades afetadas, buscando ampliar o acesso aos conhecimentos e às tecnologias desenvolvidas. Dessa forma, o avanço tecnológico deve estar acompanhado de benefícios coletivos e respeito aos interesses das partes envolvidas.

### Sustentabilidade tecnológica

Na dimensão ambiental, a análise da capacidade energética, da carga disponível, do consumo previsto e das perdas contribui para o planejamento do uso de energia. Essa abordagem se alinha à TI Verde ao permitir identificar oportunidades de eficiência, sem comprometer as margens de segurança. Entretanto, a sustentabilidade também exige considerar o ciclo de vida dos equipamentos, desde a extração de materiais e a fabricação até a manutenção e o descarte. Práticas de reparo, reutilização segura e reciclagem, associadas à economia circular, podem reduzir desperdícios e a geração de resíduos.

### Conclusão

Por fim, é necessário reconhecer os limites do projeto: calcular o consumo e as perdas energéticas não comprova que toda a missão seja sustentável. Afirmações sobre redução de impactos exigem indicadores e comparações, evitando o greenwashing. Assim, o projeto demonstra que uma decisão tecnicamente viável deve ser acompanhada de critérios éticos, responsabilidade social e atenção ambiental. O sucesso de uma missão envolve não apenas alcançar seu objetivo, mas também justificar suas decisões, proteger as pessoas e utilizar os recursos de maneira responsável.


## 8. Conclusão

Com os parâmetros definidos, o cenário `SUCESSO_DIRETO` permanece dentro das faixas seguras até `T+60s` e retorna `PRONTO PARA DECOLAR`. O cenário `FALHA_CRITICA` interrompe a contagem em `T+40s`, coerente com a regra de seis leituras consecutivas com falha. O cenário customizado permite observar variações e correções de forma controlada.
